# 7. The MD Engine (ATDYN)

Beyond analysis, genepie can drive GENESIS's **ATDYN** engine to run energy
minimization and molecular dynamics from Python. The functions return energies and
final coordinates as NumPy arrays.

```{admonition} Subprocess isolation
:class: important
ATDYN keeps global Fortran state (FFT plans, timers) between runs, which can
accumulate across many sequential calls in one process. The `*_isolated`
variants run each simulation in a fresh subprocess, making them crash-safe — the
right choice inside a long-lived notebook kernel. We use them here.
```


In [ ]:
import os
import numpy as np
from genepie import genesis_exe
import genepie, pathlib

REPO_ROOT = pathlib.Path(genepie.__file__).resolve().parents[2]
GLYCAM = REPO_ROOT / "tests" / "regression_test" / "build" / "glycam"
HAVE_DATA = (GLYCAM / "glycam.top").exists()
print("MD input data available:", HAVE_DATA)

COMMON = dict(
    forcefield="AMBER", electrostatic="PME",
    switchdist=12.0, cutoffdist=12.0, pairlistdist=14.0,
    pme_alpha=0.34, pme_ngrid_x=64, pme_ngrid_y=64, pme_ngrid_z=64, pme_nspline=4,
    dispersion_corr="epress", boundary_type="PBC",
    box_size_x=69.5294360, box_size_y=68.0597930, box_size_z=56.2256950,
)

## Energy minimization

Steepest-descent minimization of an AMBER (GLYCAM) system.

In [ ]:
minres = None
if HAVE_DATA:
    minres = genesis_exe.run_atdyn_min_isolated(
        prmtopfile=str(GLYCAM / "glycam.top"),
        ambcrdfile=str(GLYCAM / "glycam.rst"),
        method="SD", nsteps=20, eneout_period=2, nbupdate_period=4,
        rigid_bond=False, **COMMON,
    )
    print("energy terms x records:", minres.energies.shape)
    print("total energy: start = %.1f -> end = %.1f kcal/mol"
          % (minres.energies[0, 0], minres.energies[0, -1]))
    print("converged:", minres.converged, " final gradient:", round(minres.final_gradient, 4))
else:
    print("MD input data not found; skipping minimization")

## Short MD run

A few steps of NVE dynamics. `energies` is `(n_terms, n_records)`; `energy_labels` names the terms.

In [ ]:
mdres = None
if HAVE_DATA:
    mdres = genesis_exe.run_atdyn_md_isolated(
        prmtopfile=str(GLYCAM / "glycam.top"),
        ambcrdfile=str(GLYCAM / "glycam.rst"),
        integrator="VVER", nsteps=20, timestep=0.001,
        eneout_period=2, nbupdate_period=5, iseed=314159,
        rigid_bond=True, shake_iteration=500, shake_tolerance=1.0e-10,
        water_model="WAT", ensemble="NVE", tpcontrol="NO", temperature=0,
        **COMMON,
    )
    print("energy labels:", mdres.energy_labels)
    print("final coords shape (3, natom):", mdres.final_coords.shape)
else:
    print("MD input data not found; skipping MD")

## Plot the energy terms

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
fig = go.Figure()
if minres is not None:
    for i, (name, series) in enumerate(zip(minres.energy_labels, minres.energies)):
        color = PALETTE[i % len(PALETTE)]
        fig.add_trace(go.Scatter(y=series, mode="lines+markers", name=name,
                                 line=dict(color=color, width=2.5),
                                 marker=dict(size=6, color=color)))
    fig.update_layout(
        title=dict(text="<b>Energy terms during minimization</b>", font=dict(size=18)),
        xaxis_title="record", yaxis_title="energy (kcal/mol)", template="plotly_white",
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
        hovermode="x unified", margin=dict(l=70, r=30, t=60, b=50), height=440,
    )
fig

For repeated use in scripts, the same parameters work with the in-process
`run_atdyn_md` / `run_atdyn_min`. The `*_isolated` variants shown here trade a
little startup cost for crash-safety, which is ideal for notebooks and test suites.

Supported input formats: **AMBER** (`prmtopfile`/`ambcrdfile`), **GROMACS**
(`grotopfile`/`grocrdfile`), and **CHARMM** (`psffile`/`pdbfile` + `parfile`/`strfile`).
